In [ ]:
from pathlib import Path

from env_optimiser import EnvOptimiser
from feature_extractor import MinigridFeaturesExtractor
from procedural_level import ProceduralLevel
from minigrid.wrappers import ImgObsWrapper,OneHotPartialObsWrapper
from custom_callback import CustomCallback
from sb3_contrib import RecurrentPPO
import os

In [ ]:
n_envs = 10
n_timesteps = 3000000
freq = 1000
eval_freq = max(freq // n_envs, 1)  # accounting for multiple environments

save_dir = Path("models/")
save_dir.mkdir(parents=True, exist_ok=True)

env = ProceduralLevel(difficulty=1000, max_steps=120)
optimiser = EnvOptimiser(env=env, n_envs=n_envs, wrapper_cls=[OneHotPartialObsWrapper,ImgObsWrapper], save_dir=save_dir)
vec_env_train = optimiser.build_vec_env()
policy_kwargs = {
    "features_extractor_class": MinigridFeaturesExtractor,
    "features_extractor_kwargs": {"features_dim": 256},
    "normalize_images": False,
}
model_dir = optimiser.file_path.parent  # makes use of the same folder
callback = CustomCallback(check_freq=eval_freq, save_dir=model_dir, verbose=1)
model = RecurrentPPO(policy="CnnLstmPolicy", env=vec_env_train, policy_kwargs=policy_kwargs)
model.learn(total_timesteps=n_timesteps, callback=callback, progress_bar=True)

In [ ]:
def make_env() -> ImgObsWrapper:
    """Create environment with onehot wrapper and img obs wrapper."""
    env = ProceduralLevel(render_mode="human", difficulty=1000, max_steps=100)
    env = OneHotPartialObsWrapper(env)
    return ImgObsWrapper(env)


env = make_vec_env(make_env, n_envs=1, vec_env_cls=DummyVecEnv)

# init model path so that this cell is independent of the cell above, remember to change path!
model_level_path = Path("models/run_9/model.zip")

model = RecurrentPPO.load(model_level_path, env=env)
obs = env.reset()
lstm_states = None
episode_starts = np.ones((1,), dtype=bool)

while True:
    action, lstm_states = model.predict(
        obs, state=lstm_states, episode_start=episode_starts, deterministic=True
    )

    obs, rewards, dones, info = env.step(action)
    episode_starts = dones

    if dones[0]:
        break

env.close()

